# 🏨 Hotel Curator — Aggregation & Tiering

**Goal**: 
1. Aggregate the predicted attribute labels (Cleanliness, Staff, Wi-Fi, Noise, Location) up to the Hotel level.
2. Calculate a **Signal Score** (`Positive Mentions / Total Mentions`).
3. Map the Signal Score to a discrete Tier (**Elite, Superior, Premium, Fail, Uncertain**).
4. Surface the top 3 evidence sentences for each attribute.

In [ ]:
import pandas as pd
import numpy as np

# Load our successfully labeled dataset
df = pd.read_csv('data/labeled_clauses.csv')
attrs = ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']
print(f"Loaded {len(df):,} clauses for aggregation.")

### Step 1: Assign Pseudo-Hotel IDs
Because our raw dataset (TripAdvisor) does not include explicit `hotel_id` metadata, we bypass noisy Named Entity Recognition (NER) and assign a pseudo-hotel ID by clustering every 20 reviews into a single 'Hotel Entity'.

In [ ]:
# Group every 20 reviews into 1 "Hotel"
df['hotel_id'] = df['review_idx'] // 20
total_hotels = df['hotel_id'].nunique()
print(f"Simulated {total_hotels:,} unique hotels in our dataset.")

### Step 2: The Aggregation Engine
We calculate the Tiers based on the following threshold logic:
* **Elite:** $\ge 0.80$
* **Superior:** $0.60 - 0.79$
* **Premium:** $0.40 - 0.59$
* **Fail:** $< 0.40$
* **Uncertain:** Fewer than 3 mentions of the attribute.

In [ ]:
def calculate_tier(pos_count, neg_count):
    total = pos_count + neg_count
    if total < 3:
        return 'Uncertain', None
    
    score = pos_count / total
    if score >= 0.80:
        return 'Elite', score
    elif score >= 0.60:
        return 'Superior', score
    elif score >= 0.40:
        return 'Premium', score
    else:
        return 'Fail', score

def aggregate_hotel(hotel_df):
    profile = {}
    for attr in attrs:
        # Filter to clauses that actually mention this attribute
        mentions = hotel_df[hotel_df[attr] != 'not_mentioned']
        
        pos_count = (mentions[attr] == 'positive').sum()
        neg_count = (mentions[attr] == 'negative').sum()
        
        tier, score = calculate_tier(pos_count, neg_count)
        
        # Extract Top 3 Evidence sentences (we prioritize negative evidence if the tier is poor)
        if tier in ['Fail', 'Premium']:
            evidence_df = mentions[mentions[attr] == 'negative']
        else:
            evidence_df = mentions[mentions[attr] == 'positive']
            
        evidence = evidence_df['clause'].head(3).tolist()
        
        profile[attr] = {
            'tier': tier,
            'score': round(score, 2) if score is not None else "N/A",
            'evidence': evidence,
            'mentions': len(mentions)
        }
    return profile

print("Aggregation Engine Ready!")

### Step 3: Execute Pipeline & View Sample Profile

In [ ]:
print("Aggregating all 1,025 hotels...")
hotel_profiles = {}
for hotel_id, group in df.groupby('hotel_id'):
    hotel_profiles[hotel_id] = aggregate_hotel(group)

print("\n======================================================")
print("🏢 SAMPLE HOTEL PROFILE: HOTEL ID #0")
print("======================================================")
profile = hotel_profiles[0]

for attr, data in profile.items():
    tier_str = f"[{data['tier'].upper()}]"
    print(f"{attr.upper():<15} | Tier: {tier_str:<12} | Score: {data['score']} | Mentions: {data['mentions']}")
    if data['evidence']:
        print("  Evidence:")
        for i, ev in enumerate(data['evidence'], 1):
            print(f"    {i}. \"{ev}\"")
    print("-" * 60)